# 01 — Data Preprocessing

This notebook performs the reproducible preprocessing of the raw
experimental datasets.

## Objectives

1. Load the selected raw datasets.
2. Preserve the original raw data without modification.
3. Standardize variable names and data types.
4. Validate participant identifiers and experimental conditions.
5. Identify missing, duplicated, and invalid observations.
6. Prepare questionnaire/SAM data.
7. Prepare trial-level n-back data.
8. Integrate participant-level information when appropriate.
9. Generate reproducible interim datasets.

Raw datasets are treated as read-only.

In [3]:
import pandas as pd
import numpy as np

from src import config as cfg

from src.utils import (
    initialize_project,
    dataframe_summary,
)

from src.preprocessing import (
    standardize_column_names,
    normalize_missing_values,
    convert_to_numeric,
    filter_reaction_times,
)

initialize_project()

In [4]:
raw_files = {
    "sam": cfg.RAW_SAM_FILE,
    "nback2": cfg.RAW_NBACK2_FILE,
    "nback4": cfg.RAW_NBACK4_FILE,
}

for name, path in raw_files.items():
    print(
        f"{name:<10} "
        f"{'OK' if path.exists() else 'MISSING'} "
        f"{path.name}"
    )

sam        OK DBs_SAMs_V9_200626.csv
nback2     OK DBS_N_back_2_01072026.sav
nback4     OK n_back_4.xlsx


In [5]:
sam_raw = pd.read_csv(
    cfg.RAW_SAM_FILE
)

nback2_raw = pd.read_spss(
    cfg.RAW_NBACK2_FILE
)

nback4_raw = pd.read_excel(
    cfg.RAW_NBACK4_FILE
)

*_raw
   ↓
imutável

cópia
   ↓
preprocessing

In [6]:
sam = sam_raw.copy()

nback2 = nback2_raw.copy()

nback4 = nback4_raw.copy()

In [7]:
datasets = {
    "SAM": sam_raw,
    "N-back 2": nback2_raw,
    "N-back 4": nback4_raw,
}

initial_audit = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "missing_cells": int(
                df.isna().sum().sum()
            ),
            "duplicated_rows": int(
                df.duplicated().sum()
            ),
        }
        for name, df in datasets.items()
    ]
)

initial_audit

,dataset,rows,columns,missing_cells,duplicated_rows
0,SAM,24,35,0,0
1,N-back 2,17896,17,775,56
2,N-back 4,17839,16,17947,0


In [8]:
sam = standardize_column_names(
    sam_raw.copy()
)

nback2 = standardize_column_names(
    nback2_raw.copy()
)

nback4 = standardize_column_names(
    nback4_raw.copy()
)

Participant_Name
        ↓
participant_name

SAM_Neg_V1_mean
        ↓
sam_neg_v1_mean

R_Reaction_Time
        ↓
r_reaction_time

In [9]:
print(sam.columns.tolist())

['participant_name', 'order', 'group_id', 'sex', 'age', 'cog_reap', 'ex_sup', 'sam_neg_v1_mean', 'sam_neg_v2_mean', 'sam_neg_v3_mean', 'sam_neg_v3_1_mean', 'sam_neg_v3_2_mean', 'sam_neg_v4_mean', 'sam_neg_v5_mean', 'sam_neg_v6_mean', 'sam_neg_v6_3_mean', 'sam_neg_v6_4_mean', 'sam_neg_v7_mean', 'sam_pos_v1_mean', 'sam_pos_v2_mean', 'sam_pos_v3_mean', 'sam_pos_v3_1_mean', 'sam_pos_v3_2_mean', 'sam_pos_v4_mean', 'sam_pos_v5_mean', 'sam_pos_v6_mean', 'sam_pos_v6_3_mean', 'sam_pos_v6_4_mean', 'sam_pos_v7_mean', 'traite_result_neg', 'traite_result_pos', 'result_state_neg_post', 'result_state_pos_post', 'result_state_neg_pre', 'result_state_pos_pre']


In [10]:
print(nback4.columns.tolist())

['order', 'participant_group', 'participant_name', 'session_id', 'block_name', 'trial_name', 'event_name', 'cumulative_time', 'participant_response', 'key', 'pressed_or_released', 'correct_response', 'r_reaction_time', 'error_code', 'all_loction_trial_variable', 'cr']


Original variable: Participant_Name

Observed values: numeric identifiers

Proposed standardized variable: participant_id

Status: pending confirmation

In [11]:
COLUMN_RENAME_MAP = {
    "participant_name": "participant_id",
}

In [12]:
nback4["cr"].value_counts(
    dropna=False
)

cr
Correct        15084
Incorrect       2720
Not Respond       18
NaN               16
                   1
Name: count, dtype: int64

In [13]:
nback4["error_code"].value_counts(
    dropna=False
)

error_code
C      17795
NR        19
E         18
SC         4
NaN        3
Name: count, dtype: int64

In [14]:
nback4[
    "block_name"
].value_counts(
    dropna=False
)

block_name
n_back2    5958
n_back3    5941
n_back1    5940
Name: count, dtype: int64

In [15]:
nback4[
    "session_id"
].value_counts(
    dropna=False
)

session_id
1st    9315
2nd    8524
Name: count, dtype: int64

In [16]:
rt = pd.to_numeric(
    nback4["r_reaction_time"],
    errors="coerce",
)

rt.describe(
    percentiles=[
        .01,
        .05,
        .25,
        .50,
        .75,
        .95,
        .99,
    ]
)

count     810.000000
mean      661.025926
std       659.452438
min        15.000000
1%         31.360000
5%        123.450000
25%       277.000000
50%       423.500000
75%       810.000000
95%      1903.950000
99%      3166.820000
max      5789.000000
Name: r_reaction_time, dtype: float64

In [17]:
from src.config import (
    RT_MIN_MS,
    RT_MAX_MS,
)

print(
    "RT < 100 ms:",
    (rt < RT_MIN_MS).sum()
)

print(
    "RT > 5000 ms:",
    (rt > RT_MAX_MS).sum()
)

RT < 100 ms: 31
RT > 5000 ms: 1


SAM raw
   ↓
SAM preprocessing
   ↓
sam_long


N-back 2 raw
   ↓
N-back preprocessing
   ↓
nback2_trials


N-back 4 raw
   ↓
N-back preprocessing
   ↓
nback4_trials

In [18]:
def preprocess_sam(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Clean and standardize the participant-level SAM/questionnaire dataset.
    """

    df = data.copy()

    # --------------------------------------------------------------
    # Standardize structure
    # --------------------------------------------------------------

    df = standardize_column_names(df)
    df = normalize_missing_values(df)

    df = standardize_participant_id(
        df,
        source_column="participant_name",
    )

    # --------------------------------------------------------------
    # Numeric columns
    # --------------------------------------------------------------

    numeric_columns = [
        column
        for column in df.columns
        if (
            column.startswith("sam_")
            or column.startswith("result_state")
            or column.startswith("traite_result")
            or column in {
                "age",
                "cog_reap",
                "ex_sup",
                "order",
                "group_id",
                "sex",
            }
        )
    ]

    for column in numeric_columns:

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    # --------------------------------------------------------------
    # Explicit categorical columns
    # --------------------------------------------------------------

    df["group_id"] = (
        df["group_id"]
        .astype("Int64")
    )

    df["order"] = (
        df["order"]
        .astype("Int64")
    )

    df["sex"] = (
        df["sex"]
        .astype("Int64")
    )

    # --------------------------------------------------------------
    # Duplicate participant check
    # --------------------------------------------------------------

    if df["participant_id"].duplicated().any():

        duplicated_ids = (
            df.loc[
                df["participant_id"].duplicated(
                    keep=False
                ),
                "participant_id",
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Duplicated participant IDs found "
            f"in SAM dataset: {duplicated_ids}"
        )

    return df

In [19]:
def repair_nback_response_columns(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Repair datasets in which Correct_Response and
    R_Reaction_Time were inconsistently exported.

    The numeric value is interpreted as reaction time.
    The non-numeric value is interpreted as the correct-response code.
    """

    df = data.copy()

    required_columns = {
        "correct_response",
        "r_reaction_time",
    }

    missing = (
        required_columns
        - set(df.columns)
    )

    if missing:
        raise ValueError(
            "Missing required columns: "
            f"{sorted(missing)}"
        )

    correct_numeric = pd.to_numeric(
        df["correct_response"],
        errors="coerce",
    )

    rt_numeric = pd.to_numeric(
        df["r_reaction_time"],
        errors="coerce",
    )

    # --------------------------------------------------------------
    # Reconstruct reaction time
    # --------------------------------------------------------------

    df["reaction_time_ms"] = (
        rt_numeric.combine_first(
            correct_numeric
        )
    )

    # --------------------------------------------------------------
    # Reconstruct expected response/key
    # --------------------------------------------------------------

    correct_is_numeric = (
        correct_numeric.notna()
    )

    df["correct_response_clean"] = np.where(
        correct_is_numeric,
        df["r_reaction_time"],
        df["correct_response"],
    )

    df["correct_response_clean"] = (
        df["correct_response_clean"]
        .astype("string")
        .str.strip()
    )

    return df

In [20]:

def preprocess_nback(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Clean and standardize trial-level n-back data.
    """

    df = data.copy()

    # --------------------------------------------------------------
    # Basic normalization
    # --------------------------------------------------------------

    df = standardize_column_names(df)
    df = normalize_missing_values(df)

    df = standardize_participant_id(
        df,
        source_column="participant_name",
    )

    # --------------------------------------------------------------
    # Repair exported RT / correct-response columns
    # --------------------------------------------------------------

    df = repair_nback_response_columns(
        df
    )

    # --------------------------------------------------------------
    # Normalize strings
    # --------------------------------------------------------------

    string_columns = [
        "participant_group",
        "session_id",
        "block_name",
        "trial_name",
        "event_name",
        "participant_response",
        "key",
        "pressed_or_released",
        "error_code",
        "cr",
    ]

    for column in string_columns:

        if column in df.columns:

            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
            )

    # --------------------------------------------------------------
    # Numeric variables
    # --------------------------------------------------------------

    numeric_columns = [
        "order",
        "cumulative_time",
        "reaction_time_ms",
    ]

    for column in numeric_columns:

        if column in df.columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    # --------------------------------------------------------------
    # Binary accuracy
    # --------------------------------------------------------------

    if "cr" in df.columns:

        df["accuracy"] = (
            df["cr"]
            .str.lower()
            .map(
                {
                    "correct": 1,
                    "incorrect": 0,
                    "not respond": 0,
                }
            )
            .astype("Int64")
        )

    return df

In [21]:
def derive_nback_level(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Extract n-back level from trial_name.
    """

    df = data.copy()

    if "trial_name" not in df.columns:
        raise ValueError(
            "'trial_name' column not found."
        )

    df["nback_level"] = (
        df["trial_name"]
        .astype("string")
        .str.extract(
            r"n\s*=\s*(\d+)",
            expand=False,
        )
    )

    df["nback_level"] = pd.to_numeric(
        df["nback_level"],
        errors="coerce",
    ).astype("Int64")

    return df

In [22]:
def derive_session_number(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert session labels into ordered session numbers.
    """

    df = data.copy()

    mapping = {
        "1st": 1,
        "2nd": 2,
    }

    df["session_number"] = (
        df["session_id"]
        .map(mapping)
        .astype("Int64")
    )

    return df

01_preprocessing.ipynb

1. Initialization
2. Load raw datasets

3. Raw data audit
   ├── dimensions
   ├── duplicates
   ├── missing values
   └── participant counts

4. SAM preprocessing
   ├── column normalization
   ├── missing values
   ├── numeric conversion
   ├── participant IDs
   └── integrity checks

5. N-back 2 preprocessing
   ├── column normalization
   ├── missing values
   ├── numeric conversion
   ├── participant IDs
   └── integrity checks

6. N-back 4 preprocessing
   ├── column normalization
   ├── repair RT / Correct_Response
   ├── accuracy
   ├── session
   ├── n-back level
   ├── error codes
   └── RT flags

7. Cross-dataset participant audit

8. Save interim datasets

9. Preprocessing report

In [23]:
import numpy as np
import pandas as pd

from src import config as cfg

from src.utils import (
    initialize_project,
)

from src.preprocessing import (
    preprocess_sam,
    preprocess_nback,
    derive_nback_level,
    derive_session_number,
)

initialize_project()

In [24]:
sam_raw = pd.read_csv(
    cfg.RAW_SAM_FILE
)

nback2_raw = pd.read_spss(
    cfg.RAW_NBACK2_FILE
)

nback4_raw = pd.read_excel(
    cfg.RAW_NBACK4_FILE
)

In [25]:
print(
    "SAM:",
    sam_raw.shape
)

print(
    "N-back 2:",
    nback2_raw.shape
)

print(
    "N-back 4:",
    nback4_raw.shape
)

SAM: (24, 35)
N-back 2: (17896, 17)
N-back 4: (17839, 16)


In [26]:
sam = preprocess_sam(
    sam_raw
)

sam.head()

,participant_id,order,group_id,sex,age,cog_reap,ex_sup,sam_neg_v1_mean,sam_neg_v2_mean,sam_neg_v3_mean,...,sam_pos_v6_mean,sam_pos_v6_3_mean,sam_pos_v6_4_mean,sam_pos_v7_mean,traite_result_neg,traite_result_pos,result_state_neg_post,result_state_pos_post,result_state_neg_pre,result_state_pos_pre
0,1101,1,2,2,26,30.0,11.0,6.0,5.0,5.0,...,7,5,5,5,10.0,12,13.0,9,13.0,8
1,1106,2,4,2,26,23.0,16.0,5.0,4.0,2.0,...,8,6,6,6,17.0,14,12.0,11,12.0,11
2,1107,2,3,2,28,26.0,11.0,6.0,5.0,3.0,...,8,7,5,4,16.0,16,18.0,12,12.0,12
3,1108,1,2,2,25,26.0,15.0,5.0,3.0,3.0,...,5,5,5,4,11.0,10,12.0,12,12.0,12
4,1122,1,2,1,24,37.0,14.0,9.0,7.0,6.0,...,8,4,4,4,20.0,20,9.0,10,10.0,14


In [27]:
sam.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   participant_id         24 non-null     Int64  
 1   order                  24 non-null     Int64  
 2   group_id               24 non-null     Int64  
 3   sex                    24 non-null     Int64  
 4   age                    24 non-null     int64  
 5   cog_reap               23 non-null     float64
 6   ex_sup                 23 non-null     float64
 7   sam_neg_v1_mean        23 non-null     float64
 8   sam_neg_v2_mean        23 non-null     float64
 9   sam_neg_v3_mean        23 non-null     float64
 10  sam_neg_v3_1_mean      23 non-null     float64
 11  sam_neg_v3_2_mean      23 non-null     float64
 12  sam_neg_v4_mean        23 non-null     float64
 13  sam_neg_v5_mean        22 non-null     float64
 14  sam_neg_v6_mean        23 non-null     float64
 15  sam_neg_v6_3_mean  

In [28]:
sam.isna().sum()

participant_id           0
order                    0
group_id                 0
sex                      0
age                      0
cog_reap                 1
ex_sup                   1
sam_neg_v1_mean          1
sam_neg_v2_mean          1
sam_neg_v3_mean          1
sam_neg_v3_1_mean        1
sam_neg_v3_2_mean        1
sam_neg_v4_mean          1
sam_neg_v5_mean          2
sam_neg_v6_mean          1
sam_neg_v6_3_mean        1
sam_neg_v6_4_mean        1
sam_neg_v7_mean          1
sam_pos_v1_mean          0
sam_pos_v2_mean          0
sam_pos_v3_mean          0
sam_pos_v3_1_mean        0
sam_pos_v3_2_mean        0
sam_pos_v4_mean          0
sam_pos_v5_mean          0
sam_pos_v6_mean          0
sam_pos_v6_3_mean        0
sam_pos_v6_4_mean        0
sam_pos_v7_mean          0
traite_result_neg        1
traite_result_pos        0
result_state_neg_post    1
result_state_pos_post    0
result_state_neg_pre     1
result_state_pos_pre     0
dtype: int64

In [29]:
print(
    "Participants:",
    sam["participant_id"].nunique()
)

print(
    sorted(
        sam["participant_id"]
        .dropna()
        .tolist()
    )
)

Participants: 24
[1101, 1106, 1107, 1108, 1122, 1123, 1126, 1128, 1135, 1140, 1150, 1151, 1156, 1157, 1159, 1162, 1170, 1171, 1207, 1208, 1214, 1229, 1239, 1249]


In [30]:
nback4 = preprocess_nback(
    nback4_raw
)

nback4 = derive_nback_level(
    nback4
)

nback4 = derive_session_number(
    nback4
)

In [31]:
nback4[
    [
        "participant_id",
        "participant_group",
        "session_id",
        "session_number",
        "block_name",
        "trial_name",
        "nback_level",
        "participant_response",
        "correct_response_clean",
        "reaction_time_ms",
        "cr",
        "accuracy",
    ]
].head()

,participant_id,participant_group,session_id,session_number,block_name,trial_name,nback_level,participant_response,correct_response_clean,reaction_time_ms,cr,accuracy
0,1101,B,1st,1,n_back1,"LOP_n=1, 1",1,Z,/; Z,425.0,Correct,1
1,1101,B,1st,1,n_back1,"LOP_n=1, 2",1,Z,/; Z,1375.0,Correct,1
2,1101,B,1st,1,n_back1,"LOP_n=1, 3",1,Z,/; Z,523.0,Correct,1
3,1101,B,1st,1,n_back1,"LOP_n=1, 4",1,Z,/; Z,345.0,Correct,1
4,1101,B,1st,1,n_back1,"LOP_n=1, 5",1,Z,/; Z,327.0,Correct,1


In [32]:
nback4[
    "reaction_time_ms"
].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)

count    17839.000000
mean       879.224900
std       1056.968168
min          5.000000
1%          26.000000
5%         101.000000
25%        284.000000
50%        470.000000
75%       1074.000000
95%       2926.500000
99%       5174.240000
max      10007.000000
Name: reaction_time_ms, dtype: float64

In [33]:
print(
    "Original numeric RT:",
    pd.to_numeric(
        nback4_raw[
            "R_Reaction_Time"
        ],
        errors="coerce",
    ).notna().sum()
)

print(
    "Recovered numeric RT:",
    nback4[
        "reaction_time_ms"
    ].notna().sum()
)

Original numeric RT: 810
Recovered numeric RT: 17839


In [34]:
nback4[
    "cr"
].value_counts(
    dropna=False
)

cr
Correct        15084
Incorrect       2720
Not Respond       18
<NA>              17
Name: count, dtype: Int64

In [35]:
nback4[
    "accuracy"
].value_counts(
    dropna=False
)

accuracy
1       15084
0        2738
<NA>       17
Name: count, dtype: Int64

In [36]:
pd.crosstab(
    nback4["error_code"],
    nback4["cr"],
    dropna=False,
)

cr,Correct,Incorrect,Not Respond,<NA>
error_code,,,,
C,15077,2718,0,0
E,5,0,0,13
NR,1,0,18,0
SC,1,2,0,1
<NA>,0,0,0,3


In [37]:
nback4[
    "nback_level"
].value_counts(
    dropna=False
).sort_index()

nback_level
1    3961
2    5940
3    3960
4    3978
Name: count, dtype: Int64

In [38]:
pd.crosstab(
    nback4["block_name"],
    nback4["nback_level"],
)

nback_level,1,2,3,4
block_name,,,,
n_back1,1320,1980,1320,1320
n_back2,1320,1980,1320,1338
n_back3,1321,1980,1320,1320


In [39]:
participant_sessions = (
    nback4[
        [
            "participant_id",
            "participant_group",
            "session_number",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "participant_id",
            "session_number",
        ]
    )
)

participant_sessions

,participant_id,participant_group,session_number
0,1101,B,1
9315,1101,A,2
405,1106,D,1
9738,1106,C,2
810,1107,C,1
10143,1107,D,2
1215,1108,B,1
10548,1108,A,2
1620,1122,B,1
10953,1122,A,2


In [40]:
pd.crosstab(
    nback4["participant_id"],
    nback4["session_number"],
)

session_number,1,2
participant_id,,
1101,405,423
1106,405,405
1107,405,405
1108,405,405
1122,405,405
1123,405,405
1126,405,405
1128,405,405
1135,405,405


In [41]:
sam_ids = set(
    sam["participant_id"]
    .dropna()
)

nback_ids = set(
    nback4["participant_id"]
    .dropna()
)

In [42]:
print(
    "SAM participants:",
    len(sam_ids)
)

print(
    "N-back participants:",
    len(nback_ids)
)

print(
    "\nOnly in SAM:",
    sorted(
        sam_ids - nback_ids
    )
)

print(
    "\nOnly in N-back:",
    sorted(
        nback_ids - sam_ids
    )
)

SAM participants: 24
N-back participants: 23

Only in SAM: [np.int64(1140)]

Only in N-back: []


In [43]:
rt = nback4[
    "reaction_time_ms"
]

rt.describe(
    percentiles=[
        .001,
        .01,
        .05,
        .25,
        .50,
        .75,
        .95,
        .99,
        .999,
    ]
)

count    17839.000000
mean       879.224900
std       1056.968168
min          5.000000
0.1%        15.000000
1%          26.000000
5%         101.000000
25%        284.000000
50%        470.000000
75%       1074.000000
95%       2926.500000
99%       5174.240000
99.9%    10004.000000
max      10007.000000
Name: reaction_time_ms, dtype: float64

In [44]:
rt_flags = pd.Series(
    {
        "RT < 100 ms":
            (rt < 100).sum(),

        "RT < 150 ms":
            (rt < 150).sum(),

        "RT > 3000 ms":
            (rt > 3000).sum(),

        "RT > 5000 ms":
            (rt > 5000).sum(),

        "Missing RT":
            rt.isna().sum(),
    }
)

rt_flags

RT < 100 ms      865
RT < 150 ms     1477
RT > 3000 ms     839
RT > 5000 ms     201
Missing RT         0
dtype: int64

In [45]:
from src.preprocessing import (
    flag_reaction_times,
)

nback4 = flag_reaction_times(
    nback4,
    minimum_ms=cfg.RT_MIN_MS,
    maximum_ms=cfg.RT_MAX_MS,
)

In [46]:
sam.to_csv(
    cfg.INTERIM_DATA_DIR
    / "sam_clean.csv",
    index=False,
)

nback4.to_csv(
    cfg.INTERIM_DATA_DIR
    / "nback4_trials_clean.csv",
    index=False,
)

In [47]:
nback2.to_csv(
    cfg.INTERIM_DATA_DIR
    / "nback2_trials_clean.csv",
    index=False,
)

In [48]:
from src.preprocessing import (
    derive_experimental_sequence,
)

sam = derive_experimental_sequence(
    sam
)

In [49]:
sam[
    [
        "participant_id",
        "group_id",
        "category",
        "sequence",
        "group_sequence",
    ]
].sort_values(
    "participant_id"
)

,participant_id,group_id,category,sequence,group_sequence
0,1101,2,II,B_A,B to A
1,1106,4,IV,D_C,D to C
2,1107,3,III,C_D,C to D
3,1108,2,II,B_A,B to A
4,1122,2,II,B_A,B to A
5,1123,1,I,A_B,A to B
6,1126,1,I,A_B,A to B
7,1128,1,I,A_B,A to B
8,1135,4,IV,D_C,D to C
9,1140,3,III,C_D,C to D


In [50]:
audit_cr_error = pd.crosstab(
    nback4["cr"],
    nback4["error_code"],
    dropna=False,
    margins=True,
)

audit_cr_error

error_code,C,E,NR,SC,<NA>,All
cr,,,,,,
Correct,15077,5,1,1,0,15084
Incorrect,2718,0,0,2,0,2720
Not Respond,0,0,18,0,0,18
<NA>,0,13,0,1,3,17
All,17795,18,19,4,3,17839


In [51]:
audit_condition = (
    nback4.groupby(
        [
            "participant_group",
            "session_number",
            "block_name",
            "nback_level",
        ],
        dropna=False,
    )
    .agg(
        n_trials=("accuracy", "size"),
        n_valid_accuracy=("accuracy", "count"),
        accuracy_mean=("accuracy", "mean"),
        mean_rt=("reaction_time_ms", "mean"),
        valid_rt=("rt_valid", "sum"),
    )
    .reset_index()
)

audit_condition

,participant_group,session_number,block_name,nback_level,n_trials,n_valid_accuracy,accuracy_mean,mean_rt,valid_rt
0,A,1,n_back1,1,180,180,0.994444,804.833333,177
1,A,1,n_back1,2,270,270,0.877778,1564.637037,257
2,A,1,n_back1,3,180,180,0.705556,1716.266667,170
3,A,1,n_back1,4,180,180,0.738889,2014.566667,156
4,A,1,n_back2,1,180,180,1.0,841.666667,166
...,...,...,...,...,...,...,...,...,...
91,D,2,n_back2,4,120,120,0.666667,495.633333,115
92,D,2,n_back3,1,121,120,0.916667,489.280992,113
93,D,2,n_back3,2,180,180,0.844444,377.338889,154
94,D,2,n_back3,3,120,120,0.783333,419.991667,112


In [52]:
rt_audit = pd.DataFrame(
    {
        "n": [
            len(nback4),
            nback4["rt_valid"].sum(),
            (~nback4["rt_valid"]).sum(),
            nback4["rt_too_fast"].sum(),
            nback4["rt_too_slow"].sum(),
        ],
    },
    index=[
        "Total trials",
        "Valid RT",
        "Invalid RT",
        "Too fast",
        "Too slow",
    ],
)

rt_audit

,n
Total trials,17839
Valid RT,16773
Invalid RT,1066
Too fast,865
Too slow,201


In [53]:
nback_accuracy = (
    nback4.loc[
        nback4["accuracy"].notna()
    ]
    .copy()
)

In [54]:
print(
    "Accuracy dataset:",
    nback_accuracy.shape
)

print(
    nback_accuracy[
        "accuracy"
    ].value_counts()
)

Accuracy dataset: (17822, 24)
accuracy
1    15084
0     2738
Name: count, dtype: Int64


In [55]:
nback_rt = (
    nback4.loc[
        (nback4["accuracy"] == 1)
        & (nback4["rt_valid"])
    ]
    .copy()
)

In [56]:
print(
    "RT analytical trials:",
    len(nback_rt)
)

print(
    "Participants:",
    nback_rt[
        "participant_id"
    ].nunique()
)

RT analytical trials: 14186
Participants: 23


In [57]:
participant_audit = (
    nback4.groupby(
        "participant_id"
    )
    .agg(
        total_trials=(
            "accuracy",
            "size",
        ),
        valid_accuracy=(
            "accuracy",
            "count",
        ),
        correct_trials=(
            "accuracy",
            "sum",
        ),
        valid_rt=(
            "rt_valid",
            "sum",
        ),
    )
)

In [58]:
participant_audit[
    "accuracy_rate"
] = (
    participant_audit[
        "correct_trials"
    ]
    /
    participant_audit[
        "valid_accuracy"
    ]
)

participant_audit

,total_trials,valid_accuracy,correct_trials,valid_rt,accuracy_rate
participant_id,,,,,
1101,828,812,713,785,0.878079
1106,810,810,678,745,0.837037
1107,810,810,706,788,0.871605
1108,810,810,738,747,0.911111
1122,810,810,667,775,0.823457
1123,810,810,693,768,0.855556
1126,810,810,720,769,0.888889
1128,810,810,647,743,0.798765
1135,810,810,728,722,0.898765


In [60]:
analysis = nback4.merge(
    sam,
    on="participant_id",
    how="left",
    validate="many_to_one",
)

RAW
 │
 ├── SAM
 │
 └── N-back original
        │
        ▼
01_preprocessing
        │
        ├── sam_clean
        └── nback_trials_clean
                 │
                 ▼
02_data_processing
        │
        ├── derivar emotion_condition
        ├── derivar intervention
        ├── harmonizar sessões
        ├── SAM wide → long
        ├── integrar SAM + n-back
        ├── criar dataset accuracy
        ├── criar dataset RT
        └── criar block-level dataset
                 │
                 ▼
data/processed/
        │
        ├── participant_level.csv
        ├── sam_long.csv
        ├── nback_trials.csv
        ├── nback_accuracy.csv
        ├── nback_rt.csv
        └── analysis_dataset.csv
                 │
                 ▼
03_exploratory_analysis

In [62]:
analysis.to_csv(
    cfg.PROCESSED_DATA_DIR
    / "analysis.csv",
    index=False,
)

# Check trials

In [63]:
trial_key = [
    "participant_id",
    "session_number",
    "block_name",
    "trial_name",
]

trial_duplicates = (
    analysis
    .groupby(trial_key)
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

trial_duplicates

,participant_id,session_number,block_name,trial_name,n
662,1101,2,n_back2,"LOP_n=4, 25",6
663,1101,2,n_back2,"LOP_n=4, 26",7
664,1101,2,n_back2,"LOP_n=4, 27",2
665,1101,2,n_back2,"LOP_n=4, 28",5
668,1101,2,n_back2,"LOP_n=4, 30",3
11216,1159,2,n_back3,"LOP_n=1, 2",2


In [64]:
duplicated_trials = analysis.merge(
    trial_duplicates[trial_key],
    on=trial_key,
    how="inner",
)

duplicated_trials[
    [
        "participant_id",
        "session_number",
        "block_name",
        "trial_name",
        "cr",
        "error_code",
        "accuracy",
        "reaction_time_ms",
    ]
].sort_values(
    trial_key
)

,participant_id,session_number,block_name,trial_name,cr,error_code,accuracy,reaction_time_ms
0,1101,2,n_back2,"LOP_n=4, 25",Correct,E,1,1486.0
1,1101,2,n_back2,"LOP_n=4, 25",<NA>,E,<NA>,910.0
2,1101,2,n_back2,"LOP_n=4, 25",<NA>,E,<NA>,385.0
3,1101,2,n_back2,"LOP_n=4, 25",<NA>,E,<NA>,1079.0
4,1101,2,n_back2,"LOP_n=4, 25",<NA>,E,<NA>,1894.0
5,1101,2,n_back2,"LOP_n=4, 25",Incorrect,SC,0,2254.0
6,1101,2,n_back2,"LOP_n=4, 26",Correct,E,1,2124.0
7,1101,2,n_back2,"LOP_n=4, 26",<NA>,E,<NA>,990.0
8,1101,2,n_back2,"LOP_n=4, 26",<NA>,E,<NA>,737.0
9,1101,2,n_back2,"LOP_n=4, 26",<NA>,E,<NA>,188.0


In [65]:
def select_valid_trial(group):
    """
    Select the analytically valid observation when multiple
    records exist for the same experimental trial.
    """

    valid = group[
        group["accuracy"].notna()
    ]

    if len(valid) == 1:
        return valid.iloc[0]

    if len(valid) > 1:
        return valid.iloc[0]

    return group.iloc[0]

In [66]:
analysis_clean = (
    analysis
    .groupby(
        trial_key,
        as_index=False,
        group_keys=False,
    )
    .apply(select_valid_trial)
    .reset_index(drop=True)
)

In [67]:
analysis_clean = (
    analysis
    .sort_values(
        by=[
            "participant_id",
            "session_number",
            "block_name",
            "trial_name",
            "accuracy",
        ],
        na_position="last",
    )
    .drop_duplicates(
        subset=trial_key,
        keep="first",
    )
    .reset_index(drop=True)
)

In [68]:
analysis_clean.groupby(
    [
        "participant_id",
        "session_number",
        "block_name",
    ]
).size().value_counts()

135    132
Name: count, dtype: int64

In [69]:
analysis_clean[
    "accuracy"
].isna().sum()

np.int64(0)

In [70]:
session_counts = (
    analysis_clean
    .groupby("participant_id")
    ["session_number"]
    .nunique()
)

complete_crossover_ids = (
    session_counts[
        session_counts == 2
    ]
    .index
)

In [71]:
analysis_clean[
    "complete_crossover"
] = (
    analysis_clean[
        "participant_id"
    ].isin(
        complete_crossover_ids
    )
)

In [72]:
analysis_clean = analysis_clean.rename(
    columns={
        "order_x": "trial_order",
        "order_y": "experimental_order",
    }
)

In [73]:
analysis_clean.to_csv(
    cfg.PROCESSED_DATA_DIR
    / "analysis_dataset.csv",
    index=False,
)